> 在线讲义：[第 6 章 · softmax（下）：为什么要先套 e^x](https://wikieden.github.io/transformer-visual/lessons/06-softmax-exp.html) · 本书《从零读懂 Transformer》

# 第 6 章 · softmax（下）：为什么要先套 $e^{x}$

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/06-softmax-exp.html) · 仓库里 `transformer-deep-dive/notes/lessons/06-softmax-exp.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 朴素版会出负数 | [2, −1, 5] ÷ 6 | `s / s.sum()` | [0.333, −0.167, 0.833] |
| 先套 $e^{x}$ 再除 | $e^{s_j} / \sum_k e^{s_k}$ | `e = torch.exp(s); e / e.sum()` | [0.047, 0.002, 0.950] |
| 自带的 softmax | 同上 | `torch.softmax(s, dim=-1)` | [2, 1, 5] → [0.047, 0.017, 0.936] |
| 取信息 | softmax 比例 × V | `torch.softmax(s, -1) @ VALUES` | [9.36, 9.83]，几乎就是小红 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
WORDS = ["小明", "苹果", "小红"]
VALUES = torch.tensor([[0.0, 10.0], [0.0, 0.0], [10.0, 10.0]])  # 第 4 章的信息向量

**函数 `softmax_by_hand`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def softmax_by_hand(s: torch.Tensor) -> torch.Tensor:
    e = torch.exp(s)
    return e / e.sum()

**函数 `naive_ratio`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def naive_ratio(s: torch.Tensor) -> torch.Tensor:
    return s / s.sum()

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
neg = torch.tensor([2.0, -1.0, 5.0])
print("原因 1：分数有负数", neg.tolist())
print("  朴素版:", [round(float(v), 3) for v in naive_ratio(neg)], "← 出现负的注意力")
print("  e^x   :", [round(float(v), 3) for v in torch.exp(neg)], " 总和", round(float(torch.exp(neg).sum()), 2))
print("  softmax:", [round(float(v), 3) for v in softmax_by_hand(neg)])
s = torch.tensor([2.0, 1.0, 5.0])

In [ ]:
print("\n原因 2：拉开差距", s.tolist())
print("  朴素版 :", [round(float(v), 3) for v in naive_ratio(s)])
print("  softmax:", [round(float(v), 3) for v in softmax_by_hand(s)])
print("  和 torch.softmax 一样:", torch.allclose(softmax_by_hand(s), torch.softmax(s, dim=-1)))
print("  '她' 朴素版 =", (naive_ratio(s) @ VALUES).tolist())
print("  '她' softmax =", [round(float(v), 2) for v in torch.softmax(s, dim=-1) @ VALUES])

In [ ]:
print("\n一次算几行：每行各自 softmax（dim=-1 = 沿最后一维，也就是每一行内部）")
rows = torch.tensor([[2.0, 1.0, 5.0], [2.0, -1.0, 5.0]])
p = torch.softmax(rows, dim=-1)
print("  ", [[round(float(v), 3) for v in r] for r in p], " 每行加起来:", p.sum(dim=-1).tolist())

## 3. 代码里才看得到的

1. **`dim=-1`：沿最后一维，也就是每一行内部分比例**。一串数时看不出区别；从第 11 章起分数是一张表，`dim` 写错就会变成按列分比例，结果全错。
2. **真正写代码时用 `torch.softmax`，不要手写**：原因在第 7 章的代码里（数字太大时手写版会坏掉）。

## 4. 自己改一改

- 把 `rows` 那一行的 `dim=-1` 改成 `dim=0`，看哪个方向加起来变成了 1。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 先套 $e^{x}$ 再分比例：负分也变正，比例不会是负的；
> 每多 1 分多乘一次 2.718，高分更高、低分更低，注意力集中到相关的词上。